# M5 — Feature Engineering

Builds the final feature matrix for model training from three sources:
- `data/interim/training_dataset.csv` — NKC + Goodreads + SCKN labels
- `data/interim/pre_cutoff_stats.csv` — pre-cutoff review aggregates (no leakage)
- `data/interim/goodreads_author_names.json` — author_id → name (for reference)

Outputs saved to `data/processed/`:
- `training_features.csv` — full feature matrix with label column retained
- `X_train.csv` — feature columns only
- `y_train.csv` — label column only

## Known limitations

- **Goodreads snapshot is from 2017.** Books with `czech_pub_year > 2017` (4,618 records) have systematically lower pre-cutoff signal than they would in reality, because Goodreads reviews added between 2017 and the actual Czech publication date are missing from the dataset.
- **Overall NKC → Goodreads match rate is 22.3%.** The remaining 77.7% of NKC records have no Goodreads features and are excluded from training; model predictions are implicitly conditioned on a book being findable in the Goodreads catalogue.
- **Pre-cutoff signal is extremely sparse.** 40.8% of training records have zero pre-cutoff ratings (median = 1 among those with any). Features derived from pre-cutoff reviews are unreliable for many books; `has_precutoff_signal` flags records with ≥ 5 ratings.


In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO    = Path("__file__").resolve().parents[1] if "__file__" in dir() else Path.cwd().parent
INTERIM = REPO / "data" / "interim"
PROC    = REPO / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

print(f"REPO    : {REPO}")
print(f"INTERIM : {INTERIM}")
print(f"PROC    : {PROC}")

REPO    : /home/firstone/Bachelors-thesis
INTERIM : /home/firstone/Bachelors-thesis/data/interim
PROC    : /home/firstone/Bachelors-thesis/data/processed


## Step 1 — Load and merge

In [2]:
train = pd.read_csv(INTERIM / "training_dataset.csv", dtype=str)
print(f"training_dataset : {len(train):,} rows, {train.columns.tolist()}")

pre = pd.read_csv(INTERIM / "pre_cutoff_stats.csv", dtype=str)
print(f"pre_cutoff_stats : {len(pre):,} rows, {pre.columns.tolist()}")

with open(INTERIM / "goodreads_author_names.json") as f:
    author_names: dict[str, str] = json.load(f)
print(f"author_names     : {len(author_names):,} entries")

training_dataset : 23,941 rows, ['nkc_id', 'oclc', 'czech_isbn', 'czech_title', 'original_title', 'author', 'czech_pub_year', 'source_lang', 'genres', 'match_layer', 'matched_book_id', 'gr_title', 'gr_pub_year', 'gr_ratings_count', 'gr_average_rating', 'gr_text_reviews_count', 'gr_popular_shelves', 'gr_language_code', 'gr_is_ebook', 'sckn_appearances', 'sckn_bestseller']
pre_cutoff_stats : 20,185 rows, ['matched_book_id', 'czech_pub_year', 'pre_cutoff_ratings_count', 'pre_cutoff_avg_rating', 'pre_cutoff_reviews_with_text']


author_names     : 829,524 entries


In [3]:
# Left-join training dataset with pre-cutoff stats on matched_book_id.
# Books with no pre-cutoff reviews will have NaN — fill with 0.
df = train.merge(
    pre[["matched_book_id", "pre_cutoff_ratings_count", "pre_cutoff_avg_rating", "pre_cutoff_reviews_with_text"]],
    on="matched_book_id",
    how="left",
)

df["pre_cutoff_ratings_count"]      = pd.to_numeric(df["pre_cutoff_ratings_count"],     errors="coerce").fillna(0).astype(int)
df["pre_cutoff_avg_rating"]         = pd.to_numeric(df["pre_cutoff_avg_rating"],        errors="coerce")
df["pre_cutoff_reviews_with_text"]  = pd.to_numeric(df["pre_cutoff_reviews_with_text"], errors="coerce").fillna(0).astype(int)

print(f"After merge: {len(df):,} rows")
print(f"  Books with zero pre-cutoff ratings: {(df['pre_cutoff_ratings_count'] == 0).sum():,}")

After merge: 23,941 rows
  Books with zero pre-cutoff ratings: 12,147


## Step 2 — Build features

### 2a — Popularity features (5)

In [4]:
feat = pd.DataFrame(index=df.index)

# Raw pre-cutoff counts
feat["pre_cutoff_ratings_count"] = df["pre_cutoff_ratings_count"]
feat["pre_cutoff_avg_rating"]    = df["pre_cutoff_avg_rating"]

# Log-transformed count (handles heavy tail; log1p avoids log(0))
feat["log_pre_cutoff_ratings_count"] = np.log1p(df["pre_cutoff_ratings_count"])

# Binary flag: at least 5 pre-cutoff ratings → enough signal to trust the average
feat["has_precutoff_signal"] = (df["pre_cutoff_ratings_count"] >= 5).astype(int)

# Overall Goodreads ratings count — metadata only, excluded from X_train (temporal leakage)
feat["gr_ratings_count"] = pd.to_numeric(df["gr_ratings_count"], errors="coerce").fillna(0)

print("Popularity features:")
print(feat[["pre_cutoff_ratings_count", "pre_cutoff_avg_rating",
            "log_pre_cutoff_ratings_count", "has_precutoff_signal",
            "gr_ratings_count"]].describe())

Popularity features:
       pre_cutoff_ratings_count  pre_cutoff_avg_rating  \
count              23941.000000           11794.000000   
mean                   5.584353               3.860247   
std                   44.607533               0.812565   
min                    0.000000               1.000000   
25%                    0.000000               3.400000   
50%                    0.000000               4.000000   
75%                    2.000000               4.400000   
max                 4626.000000               5.000000   

       log_pre_cutoff_ratings_count  has_precutoff_signal  gr_ratings_count  
count                  23941.000000          23941.000000      2.394100e+04  
mean                       0.761584              0.158055      1.265031e+03  
std                        1.048547              0.364800      1.792123e+04  
min                        0.000000              0.000000      1.000000e+01  
25%                        0.000000              0.000000      2.5

### 2b — Shelf / genre features (7 buckets)

In [5]:
# gr_popular_shelves is stored as a JSON string list of {"name": ..., "count": ...} dicts.
# For each book, compute the share of shelf-counts in each genre bucket.

SHELF_BUCKETS = {
    "fiction":     {"fiction", "literary-fiction", "contemporary", "literary"},
    "mystery":     {"mystery", "thriller", "crime", "suspense", "detective"},
    "romance":     {"romance", "love", "chick-lit"},
    "scifi":       {"science-fiction", "sci-fi", "fantasy", "speculative-fiction"},
    "nonfiction":  {"non-fiction", "nonfiction", "biography", "memoir",
                    "history", "self-help", "true-crime"},
    "ya":          {"young-adult", "ya", "teen", "childrens", "children"},
    "classics":    {"classics", "classic", "literary-classics"},
}


def shelf_shares(shelves_json: str) -> dict[str, float]:
    """Return genre-bucket share of total shelf counts for one book."""
    try:
        shelves = json.loads(shelves_json) if isinstance(shelves_json, str) else []
    except (json.JSONDecodeError, TypeError):
        shelves = []

    bucket_totals = {b: 0 for b in SHELF_BUCKETS}
    grand_total   = 0

    for entry in shelves:
        name  = str(entry.get("name", "")).lower().replace(" ", "-")
        count = int(entry.get("count", 0) or 0)
        grand_total += count
        for bucket, keywords in SHELF_BUCKETS.items():
            if name in keywords:
                bucket_totals[bucket] += count

    if grand_total == 0:
        return {f"shelf_{b}": 0.0 for b in SHELF_BUCKETS}
    return {f"shelf_{b}": round(bucket_totals[b] / grand_total, 6) for b in SHELF_BUCKETS}


shelf_df = pd.DataFrame(
    df["gr_popular_shelves"].apply(shelf_shares).tolist(),
    index=df.index,
)
feat = pd.concat([feat, shelf_df], axis=1)

print("Shelf bucket distributions:")
print(shelf_df.describe().loc[["mean", "max"]].round(4))

Shelf bucket distributions:
      shelf_fiction  shelf_mystery  shelf_romance  shelf_scifi  \
mean         0.0224         0.0143         0.0077       0.0129   
max          0.2976         0.3925         0.3534       0.5939   

      shelf_nonfiction  shelf_ya  shelf_classics  
mean            0.0161    0.0081          0.0061  
max             0.4638    0.2748          0.3503  


### 2c — Source language dummies (4 binary)

In [6]:
# Top source languages in the NKC cohort: eng, ger, fre, rus
for lang in ["eng", "ger", "fre", "rus"]:
    feat[f"lang_{lang}"] = (df["source_lang"].str.strip().str.lower() == lang).astype(int)

lang_cols = [f"lang_{l}" for l in ["eng", "ger", "fre", "rus"]]
print("Source language counts:")
print(feat[lang_cols].sum().to_string())
print(f"\nRecords covered by the 4 dummies: {feat[lang_cols].any(axis=1).sum():,} / {len(feat):,}")

Source language counts:
lang_eng    19923
lang_ger      949
lang_fre     1055
lang_rus       64

Records covered by the 4 dummies: 21,991 / 23,941


### 2d — Temporal features (2)

In [7]:
feat["czech_pub_year"] = pd.to_numeric(df["czech_pub_year"], errors="coerce").astype("Int64")

print("Year range:", feat["czech_pub_year"].min(), "–", feat["czech_pub_year"].max())
print("Year distribution (sampled):")
print(feat["czech_pub_year"].describe())

Year range: 2003 – 2026
Year distribution (sampled):
count        23941.0
mean     2013.567854
std         6.117709
min           2003.0
25%           2009.0
50%           2013.0
75%           2018.0
max           2026.0
Name: czech_pub_year, dtype: Float64


### 2e — Attach label

In [8]:
feat["sckn_bestseller"] = (df["sckn_bestseller"].astype(str).str.lower() == "true").astype(int)

print("Feature matrix shape:", feat.shape)
print("Columns:", feat.columns.tolist())

Feature matrix shape: (23941, 18)
Columns: ['pre_cutoff_ratings_count', 'pre_cutoff_avg_rating', 'log_pre_cutoff_ratings_count', 'has_precutoff_signal', 'gr_ratings_count', 'shelf_fiction', 'shelf_mystery', 'shelf_romance', 'shelf_scifi', 'shelf_nonfiction', 'shelf_ya', 'shelf_classics', 'lang_eng', 'lang_ger', 'lang_fre', 'lang_rus', 'czech_pub_year', 'sckn_bestseller']


In [9]:
# Fixed imputation for pre_cutoff_avg_rating.
# Books with zero pre-cutoff ratings are typically obscure titles; imputing the
# global median (~4.0) would overstate their quality. Use 3.5 (mid-scale neutral).
IMPUTE_AVG_RATING = 3.5
n_imputed = feat["pre_cutoff_avg_rating"].isna().sum()
feat["pre_cutoff_avg_rating"] = feat["pre_cutoff_avg_rating"].fillna(IMPUTE_AVG_RATING)
print(f"Imputed {n_imputed:,} missing pre_cutoff_avg_rating values with {IMPUTE_AVG_RATING}")

Imputed 12,147 missing pre_cutoff_avg_rating values with 3.5


## Step 3 — Sanity checks

In [10]:
# Shape
n_rows, n_cols = feat.shape
METADATA_COLS = {"sckn_bestseller", "gr_ratings_count"}
feature_cols = [c for c in feat.columns if c not in METADATA_COLS]
print(f"Rows          : {n_rows:,}")
print(f"Feature cols  : {len(feature_cols)}: {feature_cols}")
print(f"Label col     : sckn_bestseller")
print(f"Metadata cols : gr_ratings_count (training_features.csv only)")

Rows          : 23,941
Feature cols  : 16: ['pre_cutoff_ratings_count', 'pre_cutoff_avg_rating', 'log_pre_cutoff_ratings_count', 'has_precutoff_signal', 'shelf_fiction', 'shelf_mystery', 'shelf_romance', 'shelf_scifi', 'shelf_nonfiction', 'shelf_ya', 'shelf_classics', 'lang_eng', 'lang_ger', 'lang_fre', 'lang_rus', 'czech_pub_year']
Label col     : sckn_bestseller
Metadata cols : gr_ratings_count (training_features.csv only)


In [11]:
# Missing values
print("Missing values per column:")
missing = feat[feature_cols].isnull().sum()
missing_pct = missing / n_rows * 100
missing_report = pd.DataFrame({"missing": missing, "pct": missing_pct.round(2)})
print(missing_report[missing_report["missing"] > 0].to_string())

for col in feature_cols:
    pct = feat[col].isnull().mean() * 100
    if pct > 5:
        print(f"  WARNING: {col} has {pct:.1f}% missing — consider imputation or dropping")

Missing values per column:
Empty DataFrame
Columns: [missing, pct]
Index: []


In [12]:
# Class distribution
n_pos = feat["sckn_bestseller"].sum()
n_neg = n_rows - n_pos
print(f"Class distribution:")
print(f"  Bestseller (1): {n_pos:>6,}  ({n_pos/n_rows:.1%})")
print(f"  Non-best.  (0): {n_neg:>6,}  ({n_neg/n_rows:.1%})")
print(f"  Imbalance ratio: {n_neg/n_pos:.1f}:1")

Class distribution:
  Bestseller (1):  1,118  (4.7%)
  Non-best.  (0): 22,823  (95.3%)
  Imbalance ratio: 20.4:1


In [13]:
# Feature–label correlations
numeric_feats = feat[feature_cols].select_dtypes(include="number").columns.tolist()
corrs = feat[numeric_feats + ["sckn_bestseller"]].corr()["sckn_bestseller"].drop("sckn_bestseller")
top10 = corrs.abs().sort_values(ascending=False).head(10)

print("Top 10 features by |correlation| with sckn_bestseller:")
print(corrs[top10.index].round(4).to_string())

Top 10 features by |correlation| with sckn_bestseller:
shelf_ya                    0.0741
czech_pub_year             -0.0469
shelf_mystery               0.0247
shelf_fiction               0.0237
shelf_scifi                 0.0134
lang_rus                   -0.0115
pre_cutoff_ratings_count    0.0106
lang_ger                   -0.0105
shelf_classics             -0.0090
shelf_nonfiction           -0.0059


## Step 4 — Save outputs

In [14]:
# training_features.csv — full matrix including gr_ratings_count as metadata
feat.to_csv(PROC / "training_features.csv", index=False)
print(f"Saved {len(feat):,} rows → training_features.csv")

# X_train.csv — features only (gr_ratings_count excluded: temporal leakage)
X = feat[feature_cols]
X.to_csv(PROC / "X_train.csv", index=False)
print(f"Saved X_train.csv  shape={X.shape}")
print(f"X_train columns ({len(X.columns)}): {X.columns.tolist()}")

# y_train.csv — label only
y = feat[["sckn_bestseller"]]
y.to_csv(PROC / "y_train.csv", index=False)
print(f"Saved y_train.csv  shape={y.shape}")

Saved 23,941 rows → training_features.csv


Saved X_train.csv  shape=(23941, 16)
X_train columns (16): ['pre_cutoff_ratings_count', 'pre_cutoff_avg_rating', 'log_pre_cutoff_ratings_count', 'has_precutoff_signal', 'shelf_fiction', 'shelf_mystery', 'shelf_romance', 'shelf_scifi', 'shelf_nonfiction', 'shelf_ya', 'shelf_classics', 'lang_eng', 'lang_ger', 'lang_fre', 'lang_rus', 'czech_pub_year']
Saved y_train.csv  shape=(23941, 1)
